# Registro da decisão manual da fonte de ground truth para treino (estágio 04)

O sistema não elege fonte alguma: `ground_truth.chosen_source` é uma decisão do pesquisador, apoiada pelo diagnóstico comparativo do estágio 03. Todas as fontes habilitadas geram máscara e conjunto de patches próprios; a escolha determina apenas qual conjunto será usado no treinamento. Este estágio lê e valida a escolha, exibe a base métrica do diagnóstico e persiste o relatório de registro em `MyDrive/tcc/artifacts/metrics/ground_truth/`.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace, adiciona-o ao `sys.path` e, no Colab, cria o espelho `MyDrive/tcc/repo/` (idempotente).

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")


## Dependências pinadas

Instala as versões fixadas em `requirements-runtime.txt` (incluindo rasterio, usado na leitura das máscaras), garantindo o mesmo conjunto de bibliotecas nas duas plataformas.

In [ ]:
# Instala as versões pinadas do requirements-runtime.txt no ambiente atual.
import subprocess
import sys

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)
print("Dependências instaladas a partir de:", requirements)


## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolves os caminhos canônicos de armazenamento a partir do `src/config.yaml`.

In [ ]:
# Resolve o contexto de runtime: plataforma, armazenamento canônico e configuração.
from src import io
from src.setup import initialize

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
print(f"Plataforma: {platform}")
print(f"Relatório da decisão: {storage_paths['artifacts_metrics_ground_truth']}")
print(f"Patches por fonte: {storage_paths['data_processed_patches']}/<fonte>")


## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, além de registrar as versões dos pacotes principais do ambiente, garantindo o mesmo protocolo nas duas plataformas.

In [ ]:
# Fixa sementes, flags determinísticas e registra as versões do ambiente.
from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config)
log_environment()


## Fonte de ground truth para treinamento (decisão manual)

Lê a fonte escolhida em `ground_truth.chosen_source` do `src/config.yaml`. O sistema não elege fonte alguma — a escolha é do pesquisador, apoiada pelo diagnóstico comparativo do estágio 03. Todas as fontes habilitadas possuem máscara e patches próprios; a escolha determina apenas qual conjunto será usado no treinamento (estágios 09/10).

In [ ]:
# Lê e valida a fonte de ground truth escolhida manualmente na configuração.
from src.data.mask_utils import reference_year
from src.data.source_decision import chosen_source

chosen = chosen_source()
print(f"Fonte de ground truth para treinamento: {chosen} (ano {reference_year(chosen)})")


## Base da decisão — relatório comparativo (estágio 03)

Carrega o relatório comparativo persistido no estágio 03 (reutilizado, nunca recalculado) e exibe a base métrica da decisão para a fonte escolhida: áreas de café por fonte e a concordância dos pares que envolvem a escolha.

In [ ]:
# Carrega o relatório do estágio 03 e exibe a base métrica da decisão.
from src.data.source_decision import load_comparison_report

comparison = load_comparison_report(storage_paths)
for name in comparison["sources"]:
    print(
        f"  Área de café {name} ({comparison['years'][name]}): "
        f"{comparison['area_km2'][name]:.2f} km² "
        f"({comparison['coffee_share'][name]:.2%} do AOI)"
    )
for pair_key, pair in comparison["pairs"].items():
    if chosen not in pair_key:
        continue
    m = pair["metrics"]
    print(
        f"Concordância ({pair_key}): acordo {m['overall_agreement']:.2%} | "
        f"IoU {m['iou']:.2%} | F1 {m['f1']:.2%} | kappa {m['kappa']:.2f}"
    )


## Dependência da máscara da fonte escolhida (estágio 02)

Verifica que a máscara binária da fonte escolhida, gerada no estágio 02, está disponível no caminho canônico — sem esta dependência o conjunto de patches da fonte não pode ser construído.

In [ ]:
# Verifica se a máscara da fonte escolhida (estágio 02) está disponível.
from src.data.mask_utils import source_mask_path
from src.utils import check_dependencies

chosen_mask_path = source_mask_path(chosen, storage_paths)
check_dependencies({"Máscara da fonte escolhida (estágio 02)": chosen_mask_path})


## Relatório da decisão

Persiste o relatório JSON da decisão em `MyDrive/tcc/artifacts/metrics/ground_truth/`, registrando a fonte escolhida, o ano de referência e a base métrica do diagnóstico — sem qualquer seleção automática. Execuções repetidas reutilizam o relatório existente (idempotência).

In [ ]:
# Persiste o relatório da decisão manual (reutiliza se já existir).
from src.data.source_decision import save_source_decision_report

report_path = save_source_decision_report(chosen, comparison, storage_paths)


## Resumo da etapa

Exibe o resumo do registro da decisão: fonte escolhida (decisão manual), ano de referência, área de café, fontes comparadas e o caminho do relatório persistido.

In [ ]:
# Exibe o resumo da etapa de registro da decisão.
summary = {
    "Fonte escolhida (decisão manual)": chosen,
    "Ano de referência da fonte": reference_year(chosen),
    "Área de café (fonte escolhida)": f"{comparison['area_km2'][chosen]:.2f} km²",
    "Fontes comparadas (estágio 03)": ", ".join(comparison["sources"]),
    "Relatório da decisão": str(report_path),
}
from src.utils import print_summary

print_summary(summary, "04")
